In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining"

!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from model import ChessValueNet
from dataset_manager import ChunkedChessDataset
from datetime import datetime

GPU = "cuda"
EPOCHS = 20

device = torch.device(GPU if torch.cuda.is_available() else "cpu")
model = ChessValueNet().to(device)

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.002)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

scaler = GradScaler(GPU)

dataset = ChunkedChessDataset(folder_path="./train_dataset_chunks")
train_loader = DataLoader(dataset, batch_size=4096, pin_memory=True, num_workers=0)

model.train()
print(f"Starting training loop on device: {device}")

for epoch in range(1, EPOCHS + 1):
  running_loss = 0.0
  batch_count = 0

  for batch_boards, batch_scores in train_loader:
    batch_boards = batch_boards.to(device, non_blocking=True)
    batch_scores = batch_scores.to(device, non_blocking=True).float().view(-1, 1)

    optimizer.zero_grad()

    with autocast(device_type=GPU, dtype=torch.float16):
      predictions = model(batch_boards)
      loss = criterion(predictions, batch_scores)

      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()

      running_loss += loss.item()
      batch_count += 1

      if batch_count % 500 == 0:
        print(f"Epoch {epoch} | Batch {batch_count} | Current Loss: {loss.item():.4f} | Time: {datetime.now()}")

  scheduler.step()
  with open("/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/batch_results.txt", "a") as myfile:
    myfile.write(f"Epoch {epoch} Complete! Average Loss: {running_loss / batch_count:.4f}")
  print(f"=== Epoch {epoch} Complete! Average Loss: {running_loss / batch_count:.4f} ===")

save_path = "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/chess_model_weights_test.pth"
torch.save(model.state_dict(), save_path)
print("Weights successfully saved to disk!")

Starting training loop on device: cuda
Epoch 1 | Batch 500 | Current Loss: 9.4909 | Time: 2026-06-17 12:58:06.357062
Epoch 1 | Batch 1000 | Current Loss: 5.8571 | Time: 2026-06-17 12:58:25.536944
Epoch 1 | Batch 1500 | Current Loss: 3.5414 | Time: 2026-06-17 12:58:45.192870
Epoch 1 | Batch 2000 | Current Loss: 2.8310 | Time: 2026-06-17 12:59:04.693257
Epoch 1 | Batch 2500 | Current Loss: 2.7331 | Time: 2026-06-17 12:59:23.280676
Epoch 1 | Batch 3000 | Current Loss: 2.4076 | Time: 2026-06-17 12:59:43.514928
Epoch 1 | Batch 3500 | Current Loss: 2.4782 | Time: 2026-06-17 13:00:02.532982
Epoch 1 | Batch 4000 | Current Loss: 2.4199 | Time: 2026-06-17 13:00:22.834705
Epoch 1 | Batch 4500 | Current Loss: 2.5331 | Time: 2026-06-17 13:00:41.727084
Epoch 1 | Batch 5000 | Current Loss: 2.2920 | Time: 2026-06-17 13:01:01.019077
Epoch 1 | Batch 5500 | Current Loss: 2.2647 | Time: 2026-06-17 13:01:20.900419
Epoch 1 | Batch 6000 | Current Loss: 2.7304 | Time: 2026-06-17 13:01:39.922986
Epoch 1 | Batc